In [ ]:
!pip install --quiet fastapi uvicorn[standard] pyjwt \
pydantic torch torchvision diffusers transformers accelerate pillow aiofiles httpx \
nest_asyncio pyngrok pydrive2 pydantic-settings passlib bcrypt

from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_IMAGES_DIR = '/content/drive/MyDrive/rwc_bot_images'
os.makedirs(DRIVE_IMAGES_DIR, exist_ok=True)
print("✅ Dependencies installed and Drive mounted!")


In [3]:
import os
import time
import base64
import io
import uuid
import asyncio
import threading
from datetime import datetime, timedelta
from typing import Optional, Literal
import random
import nest_asyncio
nest_asyncio.apply()
import subprocess
import sys
import json
from fastapi import HTTPException

import torch
from fastapi import FastAPI, Depends, HTTPException, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.responses import HTMLResponse, FileResponse
from pydantic import BaseModel, Field, validator
from pydantic_settings import BaseSettings
from passlib.context import CryptContext
import jwt
from PIL import Image
from diffusers import StableDiffusionPipeline
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from pyngrok import ngrok, conf
from transformers import AutoModelForCausalLM, AutoTokenizer

class Settings(BaseSettings):
    APP_NAME: str = "RWC-Bot-Studio"
    AUTHOR: str = "RafalW3bCraft"
    LICENSE: str = "MIT"
    JWT_SECRET_KEY: str = base64.urlsafe_b64encode(uuid.uuid4().bytes).decode()
    JWT_ALGORITHM: str = "HS256"
    JWT_ACCESS_TOKEN_EXPIRE_MINUTES: int = 60
    MODEL_ID: str = "Lykon/DreamShaper"
    HF_TOKEN: Optional[str] = None
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    CORS_ORIGINS: list = ["*"]


settings = Settings()

security = HTTPBearer()


CHAT_HISTORY = {}
FEEDBACK_HISTORY = []


def create_access_token(data: dict, expires_delta: Optional[timedelta] = None):
    to_encode = data.copy()
    expire = datetime.utcnow() + (expires_delta or timedelta(minutes=settings.JWT_ACCESS_TOKEN_EXPIRE_MINUTES))
    to_encode.update({"exp": expire})
    encoded_jwt = jwt.encode(to_encode, settings.JWT_SECRET_KEY, algorithm=settings.JWT_ALGORITHM)
    return encoded_jwt

def decode_access_token(token: str) -> Optional[str]:
    try:
        payload = jwt.decode(token, settings.JWT_SECRET_KEY, algorithms=[settings.JWT_ALGORITHM])
        return payload.get("sub")
    except jwt.PyJWTError:
        return None

def save_to_drive(img: Image.Image) -> str:
    fname = f"{uuid.uuid4().hex}.png"
    fpath = os.path.join(DRIVE_IMAGES_DIR, fname)
    img.save(fpath)
    return fpath

def pil_to_base64(img: Image.Image, fmt: str = 'PNG') -> str:
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("ascii")

print("✅ Core setup completed!")

✅ Core setup completed!


In [ ]:
# Data Models & AI Classes
from pydantic import BaseModel, Field, field_validator
from typing import Optional, Literal

# Pydantic Data Models
# Removing Token model as authentication is being removed
# class Token(BaseModel):
#     access_token: str
#     token_type: str

class ChatRequest(BaseModel):
    type: Literal["image", "chat"]
    prompt: str = Field(..., min_length=1)
    style: Optional[str] = Field("fantasy")
    num_inference_steps: Optional[int] = Field(125, ge=1, le=1000)
    guidance_scale: Optional[float] = Field(7.5, ge=1.0, le=20.0)
    seed: Optional[int] = None

    @field_validator("prompt")
    @classmethod
    def validate_prompt(cls, v):
        if any(x in v.lower() for x in ["nsfw", "explicit", "illegal"]):
            raise ValueError("Prompt contains forbidden content")
        return v.strip()

class ChatResponse(BaseModel):
    id: str
    prompt: str
    type: Literal["image", "chat"]
    image_url: Optional[str] = None
    image_b64: Optional[str] = None
    generation_time: Optional[float] = None
    bot_reply: Optional[str] = None
    style: Optional[str] = None

class FeedbackRequest(BaseModel):
    prompt: str
    image_url: str
    rating: int = Field(..., ge=1, le=5)
    comments: Optional[str] = None

# AI Classes
class ConversationAI:
    def __init__(self):
        try:
            print("Loading conversational AI...")
            self.tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
            self.model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium").to(settings.DEVICE)
            self.chat_history_ids = None
            print("✅ Conversational AI loaded successfully!")
        except Exception as e:
            print(f"❌ Error loading conversational AI: {e}")
            self.tokenizer = None
            self.model = None

    async def get_reply(self, text):
        if not self.model or not self.tokenizer:
            return "I'm having trouble with my conversation system. Please try again later.", False

        try:
            new_input_ids = self.tokenizer.encode(text + self.tokenizer.eos_token, return_tensors='pt').to(settings.DEVICE)
            bot_input_ids = new_input_ids if self.chat_history_ids is None else torch.cat([self.chat_history_ids, new_input_ids], dim=-1)

            self.chat_history_ids = self.model.generate(
                bot_input_ids,
                max_length=1000,
                pad_token_id=self.tokenizer.eos_token_id,
                no_repeat_ngram_size=3,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                temperature=0.8
            )

            raw_reply = self.tokenizer.decode(self.chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

            if not raw_reply.strip():
                raw_reply = "I understand you, but I'm not sure how to respond. Could you try rephrasing?"

            return raw_reply, False

        except Exception as e:
            print(f"Error during conversation generation: {e}")
            self.chat_history_ids = None
            return "I encountered an issue. Please try again.", False

class AIImageGenerator:
    def __init__(self):
        self.device = settings.DEVICE
        print(f"Loading Stable Diffusion on {self.device}...")

        try:
            kwargs = {}
            if settings.HF_TOKEN:
                kwargs["use_auth_token"] = settings.HF_TOKEN

            self.pipeline = StableDiffusionPipeline.from_pretrained(
                settings.MODEL_ID,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                safety_checker=None,
                **kwargs
            ).to(self.device)

            print("✅ Stable Diffusion loaded successfully!")

        except Exception as e:
            print(f"❌ Error loading Stable Diffusion: {e}")
            self.pipeline = None

        self.styles = {
            "fantasy": ("epic fantasy portrait, ornate details, mystical atmosphere, high quality, detailed", "lowres, blurry, bad anatomy, distorted"),
            "cyberpunk": ("neon cyberpunk portrait, futuristic aesthetic, reflective surfaces, high quality, detailed", "lowres, watermark, bad anatomy, distorted"),
            "anime": ("anime style illustration, clean lineart, vibrant colors, expressive, high quality", "messy lines, lowres, bad anatomy, distorted"),
            "oil": ("oil painting style, rich texture, masterpiece quality, detailed brushwork", "blurry, digital artifacts, lowres, flat")
        }

    async def generate(self, prompt, style, steps, guidance, seed=None):
        if not self.pipeline:
            raise Exception("Image generation pipeline not available")

        prefix, negative = self.styles.get(style, ("", ""))
        full_prompt = f"{prompt}, {prefix}".strip(', ')

        gen_seed = seed or random.randint(0, 2**31-1)
        generator = torch.Generator(device=self.device).manual_seed(gen_seed)

        params = dict(
            prompt=full_prompt,
            negative_prompt=negative,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=generator,
        )

        loop = asyncio.get_running_loop()
        def run_sync():
            result = self.pipeline(**params)
            return result.images[0] if hasattr(result, 'images') else result[0]

        start = time.time()
        img = await loop.run_in_executor(None, run_sync)
        duration = time.time() - start

        fpath = save_to_drive(img)
        return img, fpath, duration

class NeuralPromptRefiner:
    def __init__(self):
        try:
            print("Loading GPT-2 for prompt refinement...")
            self.tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
            self.model = GPT2LMHeadModel.from_pretrained("gpt2").to(settings.DEVICE)
            self.model.eval()
            print("✅ GPT-2 loaded successfully!")
        except Exception as e:
            print(f"❌ Error loading GPT-2: {e}")
            self.tokenizer = None
            self.model = None

    async def refine(self, text):
        if not self.model or not self.tokenizer:
            return text

        try:
            iid = self.tokenizer.encode(text, return_tensors="pt").to(settings.DEVICE)
            with torch.no_grad():
                out = self.model.generate(iid, max_length=min(80, iid.shape[1] + 40), do_sample=True, top_p=0.95)
            return self.tokenizer.decode(out[0], skip_special_tokens=True)
        except Exception as e:
            print(f"Error in prompt refinement: {e}")
            return text

# Initialize AI components
print("🤖 Initializing AI components...")
conversation_ai = ConversationAI()
ai_generator = AIImageGenerator()
prompt_refiner = NeuralPromptRefiner()
print("✅ All AI components initialized!")

In [5]:
# FastAPI Application
from typing import Annotated
from fastapi import status, Depends, HTTPException
from datetime import timedelta
app = FastAPI(title=settings.APP_NAME)
app.add_middleware(
    CORSMiddleware,
    allow_origins=settings.CORS_ORIGINS,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/chat", response_model=ChatResponse)
async def chat_api(req: ChatRequest):
    uname = "anonymous"
    mem = CHAT_HISTORY.setdefault(uname, [])

    if req.type == "image":
        try:
            # Generate image
            context = " ".join(f"{r}: {m}" for r,m,_ in mem[-3:]) if mem else ""
            full_prompt = f"{context} {req.prompt}".strip()
            refined = await prompt_refiner.refine(full_prompt)

            img, img_path, duration = await ai_generator.generate(
                refined, req.style, req.num_inference_steps, req.guidance_scale, req.seed
            )

            img_b64 = pil_to_base64(img)
            public_path = f"/drive/{os.path.basename(img_path)}"

            # Update history
            mem.append(("user", req.prompt, None))
            mem.append(("bot_image", "Image generated!", public_path))

            return ChatResponse(
                id=str(uuid.uuid4())[:8],
                prompt=req.prompt,
                type="image",
                style=req.style,
                image_url=public_path,
                image_b64=img_b64,
                generation_time=duration,
                bot_reply="Image generated successfully!"
            )

        except Exception as e:
            print(f"Error in image generation: {e}")
            return ChatResponse(
                id=str(uuid.uuid4())[:8],
                prompt=req.prompt,
                type="image",
                bot_reply=f"Sorry, I encountered an error generating your image: {str(e)}"
            )

    elif req.type == "chat":
        try:
            # Generate chat response
            bot_reply, _ = await conversation_ai.get_reply(req.prompt)

            # Update history
            mem.append(("user", req.prompt, None))
            mem.append(("bot_chat", bot_reply, None))

            return ChatResponse(
                id=str(uuid.uuid4())[:8],
                prompt=req.prompt,
                type="chat",
                bot_reply=bot_reply
            )

        except Exception as e:
            print(f"Error in chat generation: {e}")
            return ChatResponse(
                id=str(uuid.uuid4())[:8],
                prompt=req.prompt,
                type="chat",
                bot_reply="Sorry, I encountered an error. Please try again."
            )

    else:
        raise HTTPException(status_code=400, detail="Invalid request type")

# File serving
@app.get("/drive/{imgfile}")
async def serve_drive_img(imgfile: str):
    fpath = os.path.join(DRIVE_IMAGES_DIR, imgfile)
    if not os.path.exists(fpath):
        raise HTTPException(404, "Not found")
    return FileResponse(fpath, media_type="image/png")

# Feedback endpoint
@app.post("/feedback")
async def feedback_api(fb: FeedbackRequest):
    uname = "anonymous"
    FEEDBACK_HISTORY.append({
        "user": uname,
        "prompt": fb.prompt,
        "image_url": fb.image_url,
        "rating": fb.rating,
        "comments": fb.comments,
        "when": time.time()
    })
    return {"ok": True}

print("✅ FastAPI application configured!")

✅ FastAPI application configured!


In [6]:
@app.get("/")
async def home():
    return HTMLResponse('''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <title>🤖 RWC Bot Studio - AI Art & Chat Platform</title>
    <meta name="viewport" content="width=device-width,initial-scale=1">
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            font-family: 'Segoe UI', system-ui, sans-serif;
            min-height: 100vh; display: flex; flex-direction: column;
        }
        .container { max-width: 1200px; margin: 0 auto; padding: 20px; flex: 1; display: flex; flex-direction: column; }
        .header { text-align: center; margin-bottom: 30px; color: white; }
        .header h1 { font-size: 2.5rem; margin-bottom: 10px; text-shadow: 0 2px 4px rgba(0,0,0,0.3); }
        .chat-container { background: rgba(255,255,255,0.95); border-radius: 20px; padding: 20px; flex: 1; display: flex; flex-direction: column; box-shadow: 0 10px 30px rgba(0,0,0,0.2); }
        .mode-selector { display: flex; gap: 10px; margin-bottom: 20px; justify-content: center; }
        .mode-btn { padding: 10px 20px; border: none; border-radius: 25px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; cursor: pointer; font-weight: 500; transition: transform 0.2s; }
        .mode-btn:hover { transform: translateY(-2px); }
        .mode-btn.active { background: linear-gradient(135deg, #fa709a 0%, #fee140 100%); }
        .status-bar { background: linear-gradient(90deg, #4facfe 0%, #00f2fe 100%); color: white; padding: 15px; text-align: center; border-radius: 10px; margin-bottom: 20px; font-weight: 500; }
        .messages { flex: 1; overflow-y: auto; padding: 20px; background: #f8f9fa; border-radius: 15px; margin-bottom: 20px; max-height: 400px; }
        .message { margin-bottom: 15px; padding: 15px 20px; border-radius: 18px; max-width: 80%; word-wrap: break-word; line-height: 1.5; }
        .message.user { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; margin-left: auto; border-bottom-right-radius: 5px; }
        .message.bot { background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); color: white; margin-right: auto; border-bottom-left-radius: 5px; }
        .message img { max-width: 100%; border-radius: 10px; margin-top: 10px; border: 3px solid rgba(255,255,255,0.9); }
        .input-section { display: flex; gap: 10px; align-items: flex-end; }
        .input-group { flex: 1; display: flex; flex-direction: column; gap: 10px; }
        .input-controls { display: flex; gap: 10px; align-items: center; }
        #prompt { flex: 1; padding: 15px; border: 2px solid #e0e0e0; border-radius: 25px; font-size: 16px; resize: vertical; min-height: 50px; font-family: inherit; }
        #prompt:focus { outline: none; border-color: #667eea; box-shadow: 0 0 10px rgba(102, 126, 234, 0.3); }
        #style { padding: 10px 15px; border: 2px solid #e0e0e0; border-radius: 20px; background: white; font-size: 14px; }
        #send { padding: 15px 30px; background: linear-gradient(135deg, #fa709a 0%, #fee140 100%); color: white; border: none; border-radius: 25px; font-size: 16px; font-weight: bold; cursor: pointer; transition: transform 0.2s; }
        #send:hover { transform: translateY(-2px); }
        .typing { display: none; color: #666; font-style: italic; padding: 10px; }
        .typing.show { display: block; }
        .error { background: #ff6b6b !important; color: white; }
        .success { background: #51cf66 !important; color: white; }
        @media (max-width: 768px) { .container { padding: 10px; } .header h1 { font-size: 2rem; } .input-controls { flex-direction: column; align-items: stretch; } .message { max-width: 90%; } }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🤖 RWC Bot Studio</h1>
            <p>AI-powered image generation and intelligent conversation</p>
        </div>

        <div class="chat-container">
            <div class="mode-selector">
                <button class="mode-btn active" onclick="setMode('image')">🎨 Image Generation</button>
                <button class="mode-btn" onclick="setMode('chat')">💬 AI Chat</button>
            </div>

            <div class="status-bar" id="status">Welcome to RWC Bot Studio! Select a mode and start creating.</div>
            <div class="messages" id="messages"></div>
            <div class="typing" id="typing">AI is thinking...</div>

            <div class="input-section">
                <div class="input-group">
                    <div class="input-controls">
                        <textarea id="prompt" placeholder="Describe the image you want to create or ask me anything..."></textarea>
                        <select id="style">
                            <option value="fantasy">🧙‍♂️ Fantasy</option>
                            <option value="cyberpunk">🌃 Cyberpunk</option>
                            <option value="anime">🎌 Anime</option>
                            <option value="oil">🖼️ Oil Painting</option>
                        </select>
                    </div>
                </div>
                <button id="send">Send ✨</button>
            </div>
        </div>
    </div>

    <script>
        let currentMode = 'image'; // Removed token and loginDemo as authentication is removed

        function setMode(mode) {
            currentMode = mode;
            document.querySelectorAll('.mode-btn').forEach(btn => btn.classList.remove('active'));
            event.target.classList.add('active');

            const styleSelect = document.getElementById('style');
            const prompt = document.getElementById('prompt');

            if (mode === 'image') {
                styleSelect.style.display = 'block';
                prompt.placeholder = 'Describe the image you want to create...';
                updateStatus('🎨 Image Generation Mode - Describe your creative vision!');
            } else {
                styleSelect.style.display = 'none';
                prompt.placeholder = 'Ask me anything or start a conversation...';
                updateStatus('💬 Chat Mode - Let\\'s have an intelligent conversation!');
            }
        }

        function updateStatus(message, type = '') {
            const status = document.getElementById('status');
            status.textContent = message;
            status.className = 'status-bar';
            if (type) status.classList.add(type);
        }

        function addMessage(content, sender, imageB64 = null) {
            const messages = document.getElementById('messages');
            const messageDiv = document.createElement('div');
            messageDiv.className = `message ${sender}`;

            if (imageB64) {
                messageDiv.innerHTML = `<div>${content}</div><img src="data:image/png;base64,${imageB64}" alt="Generated image">`;
            } else {
                messageDiv.textContent = content;
            }

            messages.appendChild(messageDiv);
            messages.scrollTop = messages.scrollHeight;
        }

        function showTyping() { document.getElementById('typing').classList.add('show'); }
        function hideTyping() { document.getElementById('typing').classList.remove('show'); }

        async function sendMessage() {
            // No token check needed as authentication is removed
            // if (!token) {
            //     updateStatus('❌ Please wait for authentication to complete.', 'error');
            //     return;
            // }

            const prompt = document.getElementById('prompt').value.trim();
            if (!prompt) return;

            const style = document.getElementById('style').value;

            addMessage(prompt, 'user');
            document.getElementById('prompt').value = '';

            showTyping();
            updateStatus('🤖 AI is processing your request...');

            try {
                const response = await fetch('/chat', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json'
                    },
                    body: JSON.stringify({
                        type: currentMode,
                        prompt: prompt,
                        style: currentMode === 'image' ? style : undefined,
                        num_inference_steps: 125,
                        guidance_scale: 7.5
                    })
                });

                hideTyping();

                if (response.ok) {
                    const data = await response.json();

                    if (data.type === 'image' && data.image_b64) {
                        addMessage(`🎨 Generated "${data.prompt}" in ${data.style} style (${data.generation_time.toFixed(1)}s)`, 'bot', data.image_b64);
                        updateStatus(`✅ Image generated successfully in ${data.generation_time.toFixed(1)} seconds!`, 'success');
                    } else if (data.type === 'chat' && data.bot_reply) {
                        addMessage(data.bot_reply, 'bot');
                        updateStatus('💬 Chat response generated!', 'success');
                    } else {
                        addMessage(data.bot_reply || 'Sorry, something went wrong.', 'bot');
                        updateStatus('⚠️ Partial response received.', 'error');
                    }
                } else {
                    const errorData = await response.json();
                    addMessage(`❌ Error: ${errorData.detail || 'Something went wrong'}`, 'bot');
                    updateStatus('❌ Request failed. Please try again.', 'error');
                }

            } catch (error) {
                hideTyping();
                addMessage('❌ Connection error. Please check your internet connection.', 'bot');
                updateStatus('❌ Network error. Please try again.', 'error');
            }
        }

        document.getElementById('send').onclick = sendMessage;
        document.getElementById('prompt').addEventListener('keydown', (e) => {
            if (e.key === 'Enter' && !e.shiftKey) {
                e.preventDefault();
                sendMessage();
            }
        });

        setMode('image');

        setTimeout(() => {
            addMessage('Welcome to RWC Bot Studio! I can generate amazing images and have intelligent conversations. What would you like to create today?', 'bot');
        }, 1000);
    </script>
</body>
</html>
    ''')

print("✅ Complete frontend interface created!")

✅ Complete frontend interface created!


In [ ]:
import uvicorn
from pyngrok import ngrok, conf
from google.colab import userdata

async def run_app():
    print("🚀 Starting RWC Bot Studio...")
    print("=" * 50)
    try:
        try:
            NGROK_TOKEN = userdata.get('NGROK_TOKEN')
            conf.get_default().auth_token = NGROK_TOKEN
            print("✅ Ngrok authentication configured")
        except:
            print("⚠️ No ngrok token found in secrets. Using default configuration.")

        # Start ngrok tunnel
        public_url = ngrok.connect(8000, bind_tls=True)
        print(f"🌐 Public URL: {public_url}")

    except Exception as e:
        print(f"⚠️ Ngrok setup warning: {e}")

    # Start the FastAPI server
    config = uvicorn.Config(
        app=app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )
    server = uvicorn.Server(config)
    await server.serve()

if __name__ == "__main__":
    import asyncio
    asyncio.run(run_app())
else:
    # For notebook execution
    asyncio.create_task(run_app())
